# Supplementary Fig. 1e — Self-marker dot plots for VHC and VGN subtypes

Each cluster is named after its distinguishing marker gene (e.g. `HC_Foxp2` is the VHC cluster defined by Foxp2 expression). This notebook plots, for each cluster, expression of the gene it is named after — i.e. a diagonal/staircase confirmation that the naming is self-consistent. `Slc17a7` (Vglut1) is added on the VGN side as a shared marker of the five Vglut1+ subtypes (`Lypd1 / Nkain3 / Aldh1a3 / Calb1 / Calb2`).

Column headers in the published figure are renamed in Illustrator:
- `HC_*` → `VHC-I.*` (Type I: Gsn, Agbl1, Socs3, Car8, Atp2a3, Tshz3) / `VHC-II.*` (Type II: Foxp2, Kcnv1, St3gal6)
- `neuron_*` and `neuron_Vglut1/*` → `VGN-*`

**Inputs**
- `Vestibular_VGN_VHC_merged.h5ad` — merged VGN + VHC AnnData with `all_cluster_gene_name` in `.obs`.

**Outputs**
- `figures/SupplFig1e_VHC_dotplot.pdf` — left panel (9 VHC clusters × 9 markers)
- `figures/SupplFig1e_VGN_dotplot.pdf` — right panel (13 VGN clusters × 14 markers, Slc17a7 added)

**Notes**
- Dot size = fraction of cells expressing; colour = within-gene z-scored mean expression (`standard_scale='var'`).
- **No row/column clustering** — order follows the publication layout (VHC-II first, then VHC-I; VGN non-Vglut1 first, then Vglut1+).
- **No `min_frac` filter** — every marker is shown, even where its own cluster has lower-than-typical detection.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm

from _dotplot import marker_dotmap_simple

sc.settings.verbosity = 1
sc.set_figure_params(figsize=(5, 5), dpi_save=300, dpi=100, frameon=False)
mpl.rcParams['pdf.fonttype'] = 42
plt.rcParams['font.family'] = 'Arial'

DATA_DIR = Path('..')
FIG_DIR = Path('figures')
FIG_DIR.mkdir(exist_ok=True)

cmap_wdg = mcolors.LinearSegmentedColormap.from_list(
    'white_darkgrey', cm.Greys(np.linspace(0.0, 0.70, 256))
)

## 1. Load adata and set cluster order to match published layout

In [ ]:
adata_all = sc.read_h5ad(DATA_DIR / 'Vestibular_VGN_VHC_merged.h5ad')
adata_all.X = adata_all.layers['umi'].copy()  # dot plot uses raw counts via use_raw=True

# Publication order (VHC-II first, then VHC-I; VGN non-Vglut1 first, then Vglut1+)
publication_order = [
    # VHC — Type II at the top, then Type I
    'HC_Foxp2', 'HC_Kcnv1', 'HC_St3gal6',
    'HC_Agbl1', 'HC_Socs3', 'HC_Car8', 'HC_Atp2a3', 'HC_Tshz3', 'HC_Gsn',
    # VGN — non-Vglut1 first
    'neuron_Ptgfr', 'neuron_Fxyd7', 'neuron_Meis2', 'neuron_Slc2a4',
    'neuron_Gsg1l', 'neuron_Gstm2', 'neuron_Nxph4', 'neuron_Lmo3',
    # then Vglut1+ subtypes
    'neuron_Vglut1/Lypd1', 'neuron_Vglut1/Nkain3', 'neuron_Vglut1/Aldh1a3',
    'neuron_Vglut1/Calb1', 'neuron_Vglut1/Calb2',
]

present = list(adata_all.obs['all_cluster_gene_name'].astype(str).unique())
missing = [c for c in publication_order if c not in present]
extras = [c for c in present if c not in publication_order]
if missing:
    print('WARNING: publication-order clusters not in adata:', missing)
if extras:
    print('WARNING: extra clusters in adata, appended at end:', extras)

adata_all.obs['all_cluster_gene_name'] = pd.Categorical(
    adata_all.obs['all_cluster_gene_name'].astype(str),
    categories=[c for c in publication_order if c in present] + extras,
    ordered=True,
)
print(adata_all.obs['all_cluster_gene_name'].cat.categories.tolist())

## 2. VHC dot plot — 9 clusters × 9 self-markers
Each cluster shows expression of its own naming-gene. Markers are listed in publication order; no row/col clustering, no `min_frac` filter.

In [ ]:
# Order: matches the cluster order above (Type II first, then Type I)
vhc_markers = ['Foxp2', 'Kcnv1', 'St3gal6',
               'Agbl1', 'Socs3', 'Car8', 'Atp2a3', 'Tshz3', 'Gsn']

adata_HC_ss3 = adata_all[adata_all.obs['batch'].isin(['HC_ss3'])].copy()
# drop unused categories so the dot plot only shows VHC clusters
adata_HC_ss3.obs['all_cluster_gene_name'] = (
    adata_HC_ss3.obs['all_cluster_gene_name'].cat.remove_unused_categories()
)

fig = marker_dotmap_simple(
    adata_HC_ss3, vhc_markers,
    groupby='all_cluster_gene_name',
    color_map=cmap_wdg,
    use_raw=True, standard_scale='var',
    expression_cutoff=0.0, min_frac=None,
    fontsize=12, spines=False,
    rows_cluster=False, cols_cluster=False,
)
plt.savefig(FIG_DIR / 'SupplFig1e_VHC_dotplot.pdf', dpi=600, bbox_inches='tight')
plt.show()

## 3. VGN dot plot — 13 clusters × 14 self-markers
`Slc17a7` (Vglut1) is added as a shared marker of the five Vglut1+ subtypes; it sits between Lmo3 and Lypd1 to mark the start of the Vglut1+ block.

In [ ]:
# Order: non-Vglut1 first (Ptgfr → Lmo3), then Slc17a7 as block start, then Vglut1+ markers
vgn_markers = ['Ptgfr', 'Fxyd7', 'Meis2', 'Slc2a4', 'Gsg1l', 'Gstm2', 'Nxph4', 'Lmo3',
               'Slc17a7',  # Vglut1 — shared marker of the Vglut1+ block
               'Lypd1', 'Nkain3', 'Aldh1a3', 'Calb1', 'Calb2']

adata_VGN = adata_all[adata_all.obs['batch'].isin(['neuron'])].copy()
adata_VGN.obs['all_cluster_gene_name'] = (
    adata_VGN.obs['all_cluster_gene_name'].cat.remove_unused_categories()
)

fig = marker_dotmap_simple(
    adata_VGN, vgn_markers,
    groupby='all_cluster_gene_name',
    color_map=cmap_wdg,
    use_raw=True, standard_scale='var',
    expression_cutoff=0.0, min_frac=None,
    fontsize=12, spines=False,
    rows_cluster=False, cols_cluster=False,
)
plt.savefig(FIG_DIR / 'SupplFig1e_VGN_dotplot.pdf', dpi=600, bbox_inches='tight')
plt.show()